# AquaInsight — Task 1: Feature Scaling Comparison

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


# Task 1 — Feature Scaling Comparison

### Internship requirement
Implement and compare multiple feature-scaling techniques, including **Min-Max Scaling, Standardization, and Robust Scaling**, by assessing their effects on distribution, outlier sensitivity, and downstream model suitability.

### Steps
1. Select `ResultValue` as the continuous demonstration variable.
2. Apply Min-Max Scaling.
3. Apply Standardization (Z-score).
4. Apply Robust Scaling using median and IQR.
5. Compare summary statistics and percentiles.
6. Visualize the transformed distributions.
7. Explain which scaler is most robust to outliers.

In [ ]:
# Step 3 — Apply and compare three scaling techniques

x = df["ResultValue"].dropna().astype(float)

scalers = {
    "Min-Max": MinMaxScaler(),
    "Standardization": StandardScaler(),
    "Robust": RobustScaler()
}

scaled_summary = []
for name, scaler in scalers.items():
    z = scaler.fit_transform(x.to_numpy().reshape(-1, 1)).ravel()
    scaled_summary.append({
        "method": name,
        "mean": z.mean(),
        "std": z.std(),
        "median": np.median(z),
        "p01": np.quantile(z, 0.01),
        "p99": np.quantile(z, 0.99),
        "min": z.min(),
        "max": z.max()
    })

scaling_results = pd.DataFrame(scaled_summary)
display(scaling_results.round(4))

sample_x = x.sample(min(10000, len(x)), random_state=42)

fig = plt.figure(figsize=(10, 5))
plt.boxplot([
    MinMaxScaler().fit_transform(sample_x.to_numpy().reshape(-1, 1)).ravel(),
    StandardScaler().fit_transform(sample_x.to_numpy().reshape(-1, 1)).ravel(),
    RobustScaler().fit_transform(sample_x.to_numpy().reshape(-1, 1)).ravel()
], labels=["Min-Max", "Standardization", "Robust"])
plt.title("Feature Scaling Comparison — ResultValue")
plt.ylabel("Transformed value")
plt.show()

print(
    "Interpretation: Min-Max is bounded but sensitive to extreme values; "
    "Standardization gives a z-score scale; Robust Scaling is generally "
    "less affected by extreme observations."
)


## Task 1 — Conclusion

The analysis above completes the requested **Feature Scaling Comparison** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.